# Агрегация, объединение и аналитическая витрина

**Практика слушателя. Продолжительность: 90 минут.**

Работа продолжается на каталоге `MoviesOnStreamingPlatforms.csv`. Для автономного запуска пары используется контрольный файл `movies_long_checkpoint.csv`, заранее подготовленный из исходного датасета.

**Источник датасета:** [https://github.com/Swapnajacontribution/Datasets/blob/main/MoviesOnStreamingPlatforms.csv](https://github.com/Swapnajacontribution/Datasets/blob/main/MoviesOnStreamingPlatforms.csv)


Результаты пары:

- `groupby` и именованные агрегации;
- `pivot_table`, `stack`, `unstack`;
- безопасный `merge` со справочниками;
- категориальное кодирование через `get_dummies`;
- аналитическая витрина и графики.

In [ ]:
# -----------------------------------------------------------------------------
# Назначение ячейки: настроить окружение и универсальные относительные пути.
# Код определяет, откуда запущен notebook, создает папки для результатов и проверяет
# наличие исходного CSV. Благодаря этому одинаковый файл работает в VS Code,
# обычном Jupyter и Google Colab без жестко заданных локальных путей пользователя.
# -----------------------------------------------------------------------------

from pathlib import Path
import sys
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Определяем, выполняется ли notebook в Google Colab.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Python:", sys.version.split()[0])
print("Операционная система:", platform.system())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Google Colab:", IN_COLAB)
print("Текущая рабочая папка:", Path.cwd())


def find_project_root() -> Path:
    """Ищет папку пары по наличию исходного CSV.

    Функция позволяет запускать notebook:
    - из корня папки пары;
    - из подпапки notebooks в VS Code или Jupyter;
    - в Google Colab после распаковки папки пары в /content.
    """
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    marker = Path("data/raw/MoviesOnStreamingPlatforms.csv")
    for candidate in candidates:
        if (candidate / marker).exists():
            return candidate

    # Дополнительные типовые расположения для Google Colab.
    colab_candidates = [
        Path('/content'),
        Path('/content/01_types_and_reshape'),
        Path('/content/02_aggregation_and_datamart'),
        Path('/content/03_scaling_methods'),
        Path('/content/04_reproducible_preprocessing'),
    ]
    for candidate in colab_candidates:
        if (candidate / marker).exists():
            return candidate

    raise FileNotFoundError(
        "Не найден data/raw/MoviesOnStreamingPlatforms.csv. "
        "Распакуйте архив пары целиком и запустите notebook повторно."
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "MoviesOnStreamingPlatforms.csv"
print("Корень проекта:", PROJECT_ROOT)
print("Исходный файл:", DATA_PATH)

## 1. Загрузка контрольной длинной таблицы и справочников

In [ ]:
# -----------------------------------------------------------------------------
# Назначение ячейки: загрузить длинную таблицу и два справочника.
# movies_long_checkpoint.csv позволяет запускать вторую пару независимо от первой.
# Справочник платформ добавляет характеристики платформ, а справочник возрастных
# маркировок — числовой порог и укрупненную аудиторию. Сразу проверяем размер таблиц.
# -----------------------------------------------------------------------------

long_path = RAW_DIR / "movies_long_checkpoint.csv"
platform_ref_path = RAW_DIR / "platforms_reference.csv"
age_ref_path = RAW_DIR / "age_rating_reference.csv"

movies_long = pd.read_csv(long_path)
platform_ref = pd.read_csv(platform_ref_path)
age_ref = pd.read_csv(age_ref_path)

print("Длинная таблица:", movies_long.shape)
print("Справочник платформ:", platform_ref.shape)
print("Справочник возрастных категорий:", age_ref.shape)
display(movies_long.head())
display(platform_ref)
display(age_ref)

## 2. Группировка и именованные агрегации

Одна строка `platform_summary` описывает одну платформу. Мы считаем число уникальных фильмов и характеристики рейтинга.

In [ ]:
# -----------------------------------------------------------------------------
# Назначение ячейки: получить агрегированную статистику по каждой платформе.
# groupby() разделяет строки на группы, а agg() рассчитывает несколько показателей:
# число уникальных фильмов, среднее/медиану рейтинга, средний возраст фильма и долю
# многоплатформенных фильмов. Округление выполняется только для удобства представления.
# -----------------------------------------------------------------------------

platform_summary = (
    movies_long
    .groupby("platform", as_index=False)
    .agg(
        movie_count=("movie_id", "nunique"),
        mean_rating=("rating_score", "mean"),
        median_rating=("rating_score", "median"),
        mean_movie_age=("movie_age", "mean"),
        multi_platform_share=("availability_group", lambda s: s.eq("multi_platform").mean()),
    )
    .sort_values("movie_count", ascending=False)
)

platform_summary["mean_rating"] = platform_summary["mean_rating"].round(2)
platform_summary["median_rating"] = platform_summary["median_rating"].round(2)
platform_summary["mean_movie_age"] = platform_summary["mean_movie_age"].round(2)
platform_summary["multi_platform_share"] = platform_summary["multi_platform_share"].round(4)

display(platform_summary)

## 3. Визуализация агрегированных результатов

In [ ]:
# -----------------------------------------------------------------------------
# Назначение ячейки: визуализировать агрегированные показатели.
# Горизонтальная диаграмма удобна для сравнения количества фильмов по платформам,
# а вертикальная — для сопоставления среднего рейтинга. Перед построением данные
# сортируются, чтобы порядок столбцов помогал быстрее увидеть различия.
# -----------------------------------------------------------------------------

plot_data = platform_summary.set_index("platform").sort_values("movie_count", ascending=True)
ax = plot_data["movie_count"].plot(kind="barh", figsize=(9, 4), title="Количество уникальных фильмов по платформам")
ax.set_xlabel("Количество фильмов")
ax.set_ylabel("Платформа")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "movies_by_platform.png", dpi=140)
plt.show()

rating_plot = platform_summary.set_index("platform").sort_values("mean_rating", ascending=False)
ax = rating_plot["mean_rating"].plot(kind="bar", figsize=(9, 4), title="Средний рейтинг фильмов по платформам")
ax.set_xlabel("Платформа")
ax.set_ylabel("Средний рейтинг")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mean_rating_by_platform.png", dpi=140)
plt.show()

## 4. `pivot_table`, `stack` и `unstack`

`pivot_table` применяет агрегацию, поэтому подходит для повторяющихся комбинаций «платформа — десятилетие». Обычный вызов `.stack()` используется без параметра `future_stack`, чтобы код работал и в старых версиях pandas.

In [ ]:
# -----------------------------------------------------------------------------
# Назначение ячейки: показать pivot_table(), stack() и unstack() как взаимосвязанные операции.
# pivot_table() строит матрицу «платформа × десятилетие» с числом уникальных фильмов.
# stack() переносит столбцы десятилетий в строки, unstack() возвращает их обратно.
# assert доказывает, что цикл stack → unstack не изменил значения исходной матрицы.
# -----------------------------------------------------------------------------

movies_by_decade = pd.pivot_table(
    movies_long,
    index="platform",
    columns="decade",
    values="movie_id",
    aggfunc="nunique",
    fill_value=0,
)
movies_by_decade.columns.name = "decade"
display(movies_by_decade)

# stack переносит столбцы десятилетий в строки.
stacked = movies_by_decade.stack().rename("movie_count").reset_index()
display(stacked.head(10))

# unstack возвращает десятилетия обратно в столбцы.
unstacked = (
    stacked
    .set_index(["platform", "decade"])["movie_count"]
    .unstack(fill_value=0)
)

assert movies_by_decade.sort_index(axis=1).equals(unstacked.sort_index(axis=1))

# Линейный график показывает динамику числа доступных фильмов по десятилетиям.
plot_matrix = movies_by_decade.transpose().sort_index()
ax = plot_matrix.plot(figsize=(10, 5), marker="o", title="Количество фильмов по платформам и десятилетиям")
ax.set_xlabel("Десятилетие")
ax.set_ylabel("Количество фильмов")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "movies_by_platform_and_decade.png", dpi=140)
plt.show()

## 5. `merge` и контроль кардинальности

Справочник платформ содержит одну строку на платформу. Длинная таблица содержит много строк на платформу, поэтому ожидаем связь `many_to_one`. Параметр `validate` проверяет это ожидание.

In [ ]:
# -----------------------------------------------------------------------------
# Назначение ячейки: обогатить длинную таблицу данными из справочников.
# Первый merge выполняется как many_to_one: многим фильмам одной платформы соответствует
# ровно одна строка справочника. validate и indicator защищают от размножения строк
# и позволяют обнаружить несопоставленные ключи до продолжения анализа.
# -----------------------------------------------------------------------------

rows_before = len(movies_long)

movies_enriched = movies_long.merge(
    platform_ref,
    on="platform",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Строк до merge:", rows_before)
print("Строк после merge:", len(movies_enriched))
print("Результат сопоставления:")
display(movies_enriched["_merge"].value_counts(dropna=False).to_frame("rows"))

assert len(movies_enriched) == rows_before
assert movies_enriched["_merge"].eq("both").all()
movies_enriched = movies_enriched.drop(columns=["_merge"])

# Второй справочник добавляет минимальный возраст и укрупненную аудиторию.
movies_enriched = movies_enriched.merge(
    age_ref,
    on="age_rating",
    how="left",
    validate="many_to_one",
)

display(movies_enriched.head())

## 6. Категориальное кодирование

`pd.get_dummies` удобно для прозрачной демонстрации One-Hot Encoding. Каждая категория превращается в отдельный бинарный столбец.

In [ ]:
# -----------------------------------------------------------------------------
# Назначение ячейки: преобразовать категориальные признаки в бинарные столбцы.
# pd.get_dummies() создает отдельный признак для каждой возрастной категории и группы
# доступности. Затем бинарные признаки объединяются с идентификатором фильма и платформой.
# Такой формат пригоден для дальнейшего статистического анализа и машинного обучения.
# -----------------------------------------------------------------------------

encoded_categories = pd.get_dummies(
    movies_enriched[["age_rating", "availability_group"]],
    prefix=["age", "availability"],
    dtype="int64",
)
movie_features_one_hot = pd.concat(
    [movies_enriched[["movie_id", "platform"]].reset_index(drop=True), encoded_categories.reset_index(drop=True)],
    axis=1,
)

display(movie_features_one_hot.head())

## 7. Сборка аналитической витрины

Гранулярность витрины: **платформа × десятилетие × возрастная маркировка**.

In [ ]:
# -----------------------------------------------------------------------------
# Назначение ячейки: собрать итоговую аналитическую витрину и сохранить результаты.
# Гранулярность витрины: платформа × десятилетие × возрастная маркировка.
# Для каждой комбинации рассчитываются объем, рейтинги, возраст фильмов и доля высоких
# оценок. В конце сохраняются промежуточные таблицы, one-hot признаки и итоговая витрина.
# -----------------------------------------------------------------------------

movies_datamart = (
    movies_enriched
    .groupby(["platform", "platform_code", "reporting_order", "decade", "age_rating"], as_index=False)
    .agg(
        movie_count=("movie_id", "nunique"),
        mean_rating=("rating_score", "mean"),
        median_rating=("rating_score", "median"),
        mean_movie_age=("movie_age", "mean"),
        high_rating_share=("rating_score", lambda s: s.ge(80).mean()),
    )
    .sort_values(["reporting_order", "decade", "age_rating"])
)

for column in ["mean_rating", "median_rating", "mean_movie_age", "high_rating_share"]:
    movies_datamart[column] = movies_datamart[column].round(3)

display(movies_datamart.head(15))

platform_summary.to_csv(OUTPUT_DIR / "platform_summary.csv", index=False, encoding="utf-8")
movies_by_decade.to_csv(OUTPUT_DIR / "movies_by_platform_and_decade.csv", encoding="utf-8")
movie_features_one_hot.to_csv(OUTPUT_DIR / "movie_features_one_hot.csv", index=False, encoding="utf-8")
movies_datamart.to_csv(PROCESSED_DIR / "movies_datamart.csv", index=False, encoding="utf-8")

print("Готово:", PROCESSED_DIR / "movies_datamart.csv")

## Итог пары

Объясните:

1. Почему после `merge` число строк не должно измениться?
2. Чем `pivot` отличается от `pivot_table`?
3. Что означает связь `many_to_one`?
4. Почему сумма `movie_count` по платформам может быть больше числа уникальных фильмов?